# Build Filings Index: SEC EDGAR Ingestion

This notebook **builds** the precomputed RAG artifacts that the companion demo
(`filings_rag_qa.ipynb`) queries. It ingests a filing **live from SEC EDGAR**,
parses the HTML, chunks and embeds it, and writes artifacts to
`data/processed/filings/{ISSUER}/{PERIOD}/` in the exact schema the demo expects.

Workflow is plug-and-play: **ingest once here, then query instantly there.**

The two notebooks are intentionally separate. Ingestion is slow and spends
embedding-API money; querying is fast and cheap. This notebook is the slow half.

**Pipeline:** acquire (EDGAR) → parse (HTML) → chunk → embed → write artifacts → verify.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/filings-qa-rag-demo/blob/main/notebooks/build_filings_index.ipynb)

## Colab / Local Setup

Runs on Google Colab and locally. On Colab it clones the public repository
(bringing the `genai_filings` package and existing artifacts) and installs
dependencies — including `beautifulsoup4` and `requests` for EDGAR ingestion.
Locally, it detects the existing repo and uses it in place.

In [ ]:
# --- Environment detection: Colab vs local ---
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/FranQuant/filings-qa-rag-demo"
REPO_NAME = "filings-qa-rag-demo"

if IN_COLAB:
    # Clone the repo (package + existing artifacts) and install deps.
    if not Path(REPO_NAME).exists():
        os.system(f"git clone --depth 1 {REPO_URL}")
    os.chdir(REPO_NAME)
    os.system("pip -q install openai pyarrow pandas python-dotenv beautifulsoup4 requests")
    PROJECT_ROOT = Path.cwd()
else:
    # Local: notebook lives in notebooks/, repo root is one level up.
    PROJECT_ROOT = Path("..").resolve()
    os.chdir(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Src path:", SRC_PATH)
print("Working directory:", os.getcwd())

## API Key & SEC Identification

Two things are needed:

- **OpenAI key** — embeddings use `text-embedding-3-small` (1536-dim), the same
  model recorded in the existing manifests, so the demo's retrieval still matches.
- **SEC contact** — EDGAR needs no API key, but it **requires** a `User-Agent`
  that identifies the caller (an email or company name). Requests are throttled
  to stay under SEC's 10 req/s guidance.

Locally, values are read from `.env` if present (`OPENAI_API_KEY`, optional
`SEC_CONTACT`). Otherwise you'll be prompted. Nothing is written back to disk.

In [ ]:
import os
from getpass import getpass

# Locally, load .env if it exists (no-op on Colab where there is none).
try:
    from dotenv import load_dotenv
    env_path = PROJECT_ROOT / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print("Loaded environment variables from .env")
except ImportError:
    pass

# OpenAI key is required for embeddings (and for the verification step).
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

# SEC contact is required for the EDGAR User-Agent (not a secret, just an ID).
SEC_CONTACT = os.getenv("SEC_CONTACT", "").strip()
if not SEC_CONTACT:
    SEC_CONTACT = input("SEC contact (email or company name) for the User-Agent: ").strip()

print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))
print("SEC contact set:", bool(SEC_CONTACT))

## Configure the Target Filing

**These are the two variables to change** to ingest a different company/period.
`ISSUER` is a US-filer ticker; `PERIOD` is a quarter label (year optional —
omit it to take the most recent matching quarter). 10-Qs cover Q1–Q3 only
(Q4 is folded into the annual 10-K).

Defaults target **JPMorgan (JPM), 10-Q, Q1 2026** — a US 10-Q filer with net
interest margin reported, which makes the verification query meaningful.

> Note on scope: EDGAR hosts the formal filing only — no earnings-call
> transcripts. This corpus answers on reported numbers, MD&A, and risk factors,
> not management Q&A. (Nubank/`NU` files as a foreign issuer — 20-F/6-K — which
> is why we prove the EDGAR path on a US 10-Q filer instead.)

In [ ]:
ISSUER = "JPM"        # US-filer ticker
PERIOD = "2026Q1"     # "Q1" | "2026Q1" | "Q1-2026" ... (Q1-Q3 only for 10-Q)
FORM = "10-Q"
EMBEDDING_MODEL = "text-embedding-3-small"  # must match existing manifests (1536-dim)

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "filings"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "filings"
OUT_DIR = PROCESSED_ROOT / ISSUER.upper() / PERIOD.upper()

print(f"Target: {ISSUER} {FORM} {PERIOD}")
print(f"Artifacts will be written to: {OUT_DIR}")

## Stage 1 — Acquire (SEC EDGAR)

Resolves the ticker to its CIK via `company_tickers.json`, reads the issuer's
`submissions` JSON, selects the target 10-Q by reporting period, and downloads
the primary HTML document to `data/raw/filings/{ISSUER}/{PERIOD}/`. A
`manifest.json` records full provenance (CIK, accession, report/filing dates,
SHA-256, source URL).

In [ ]:
from genai_filings.acquisition import run_edgar_fetch
from genai_filings.acquisition.edgar import build_user_agent

user_agent = build_user_agent(SEC_CONTACT)
print("User-Agent:", user_agent)

manifest = run_edgar_fetch(ISSUER, PERIOD, RAW_ROOT, user_agent, form=FORM)
doc = manifest["files"][0]
print(f"\nFetched: {doc['primary_document']}")
print(f"  form         : {doc['form']}")
print(f"  report_date  : {doc['report_date']}")
print(f"  filing_date  : {doc['filing_date']}")
print(f"  accession    : {doc['accession_number']}")
print(f"  bytes        : {doc['bytes']:,}")
print(f"  url          : {doc['url']}")

# Adopt the canonical YYYYQN period the fetch resolved (a bare "Q1" is
# normalized to e.g. "2026Q1" from the filing's report date). Downstream
# stages use this so the folder and the period column stay consistent.
if manifest.get("period_requested") and manifest["period_requested"] != manifest["period"]:
    print(f"\nPeriod normalized: {manifest['period_requested']} -> {manifest['period']}")
PERIOD = manifest["period"]
OUT_DIR = PROCESSED_ROOT / ISSUER.upper() / PERIOD
print("Canonical period:", PERIOD)

## Stage 2 — Parse (HTML → clean sections)

The 10-Q is inline-XBRL HTML, not PDF, so the PDF parser does not apply. This
strips scripts, hidden XBRL facts (`ix:hidden`/`ix:header`), and renders the
hundreds of financial tables **row by row with `|` cell delimiters** — which
keeps numbers from smushing into adjacent labels. Output is
`sections.parquet` (`issuer, period, doc_type, source_file, page, text`).

In [ ]:
from genai_filings.parsing import run_parse_html
import pyarrow.parquet as pq

sections_path = run_parse_html(ISSUER, PERIOD, RAW_ROOT, PROCESSED_ROOT)
sections_df = pq.read_table(sections_path).to_pandas()
print(f"sections.parquet rows: {len(sections_df)} -> {sections_path}")
print(f"columns: {list(sections_df.columns)}")
print("\n--- first section (preview) ---")
print(sections_df.iloc[0]["text"][:500])

## Stage 3 — Chunk

Reuses `genai_filings.indexing.run_index` unchanged: it concatenates the
document's sections in order and re-chunks into fixed token windows
(600 tokens, 100 overlap), writing `chunks.parquet`. **Inspect the chunk output
below to sanity-check parsing quality** — financial tables should be readable
and key figures preserved.

In [ ]:
from genai_filings.indexing import run_index

chunks_path = run_index(ISSUER, PERIOD, PROCESSED_ROOT, PROCESSED_ROOT)
chunks_df = pq.read_table(chunks_path).to_pandas()

print(f"Total chunks: {len(chunks_df)}")
print(f"char_len: min={chunks_df.char_len.min()} "
      f"max={chunks_df.char_len.max()} mean={int(chunks_df.char_len.mean())}")
print(f"doc_type: {chunks_df.doc_type.unique().tolist()}")

# Show a chunk that contains a financial table, to verify rendering quality.
mask = chunks_df.text.str.contains("Net interest income", case=False, na=False)
sample = chunks_df[mask].iloc[0] if mask.any() else chunks_df.iloc[0]
print(f"\n--- chunk {int(sample.chunk_id)} (financial detail) ---")
print(sample.text[:900])

## Stage 4 — Embed

Reuses `genai_filings.embeddings.run_embed` unchanged. **Embeddings use
`text-embedding-3-small` (1536-dim)** — the same model recorded in the existing
manifests — so the demo's query embeddings live in the same vector space and
retrieval matches. The `chunk_uid` is computed by the shared
`genai_filings.common` helpers, so vectors align with chunks exactly.

This is the step that spends API money (cheap for one filing: ~100K tokens).

In [ ]:
from genai_filings.embeddings import run_embed
import json

embeddings_path, emb_manifest_path = run_embed(
    ISSUER, PERIOD, PROCESSED_ROOT, PROCESSED_ROOT,
    model=EMBEDDING_MODEL, batch_size=64, force=False,
)
print(f"embeddings -> {embeddings_path}")
print(f"manifest   -> {emb_manifest_path}\n")
print(json.dumps(json.load(open(emb_manifest_path)), indent=2))

## Stage 5 — Verify with the Existing Demo

The artifacts are now in the same schema and location as the bundled `NU/2025Q2`
data. We confirm the **existing, unmodified** retrieval and synthesis code works
against them. Net interest margin is a real topic in a bank 10-Q (reported as
*net yield on interest-earning assets*), so this is a meaningful grounded test.

In [ ]:
from genai_filings.retrieval import retrieve
from genai_filings.answering import synthesize_answer
from IPython.display import Markdown, display

def _md_safe(text: str) -> str:
    # Escape $ so dollar amounts don't trigger LaTeX math rendering.
    return (text or "").replace("\\$", "$").replace("$", "\\$")

query = "How is net interest margin evolving?"

results = retrieve(query=query, issuer=ISSUER, period=PERIOD, k=5)
import pandas as pd
display(pd.DataFrame(results)[["rank", "score", "doc_type", "chunk_id", "text_preview"]])

answer = synthesize_answer(
    query=query, issuer=ISSUER, period=PERIOD, k=5,
    provider="openai", model="gpt-5.5", temperature=0.0, max_tokens=600,
)
print(
    f"provider={answer['provider']} | model={answer['model']} | "
    f"chunks={answer['used_chunks']} | "
    f"valid_cites={answer['citations_valid']} | "
    f"invalid_cites={answer['citations_invalid']}"
)
display(Markdown(_md_safe(answer["answer_markdown"])))

## Done — Now Query Instantly

The index for your chosen issuer/period is built (JPM 2026Q1 by default). To ask cited questions over it
(fast, cheap, no rebuild), open the companion demo `filings_rag_qa.ipynb` and
point it at this issuer/period:

```python
retrieve(query="...", issuer="JPM", period="2026Q1", k=5)
synthesize_answer(query="...", issuer="JPM", period="2026Q1", k=5, provider="openai")
```

To ingest another company, change `ISSUER` and `PERIOD` at the top and re-run.